In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
os.chdir("/workspaces/TFG_Spain-s_Migratory_Flow")  # forzar raíz del proyecto

SETID = Path("data/conectividad/setid")
OUT   = Path("output/conectividad/02-silver")
OUT.mkdir(parents=True, exist_ok=True)

# ── 1. ARCHIVO 2013-2020 ─────────────────────────────────────────────────────
df_old = pd.read_excel(
    SETID / "Cobertura_BA_España_2013-2020_ESP_MUN_PROV_CCAA_Nacional_datosgob.xlsx",
    sheet_name="Provincia"
)

# Columnas de interés: Cob. 100Mbps por año
cols_100_old = [c for c in df_old.columns if "Cob. 100Mbps" in c]
id_cols_old  = ["Comunidad Autónoma", "Provincia"]

df_old_sel = df_old[id_cols_old + cols_100_old].copy()

# Melt → formato largo
df_old_long = df_old_sel.melt(
    id_vars=id_cols_old,
    value_vars=cols_100_old,
    var_name="periodo",
    value_name="cob_100mbps"
)

# Extraer año del nombre de columna (e.g. "(dic 2015)" → 2015)
df_old_long["año"] = df_old_long["periodo"].str.extract(r"(\d{4})").astype(int)

# Quedar con el máximo por provincia×año (hay junio y dic en algunos años)
df_old_agg = (df_old_long
    .groupby(["Comunidad Autónoma", "Provincia", "año"], as_index=False)["cob_100mbps"]
    .max()
)

In [2]:
# ── 2. ARCHIVO 2021-2024 ─────────────────────────────────────────────────────
f_new = [f for f in sorted(SETID.glob("*.xlsx")) if "2021" in f.name][0]

df_new = pd.read_excel(f_new, sheet_name="Provincia_%viviendas")

cols_100_new = [c for c in df_new.columns if "100Mbps" in c]
id_cols_new  = ["Comunidad Autónoma", "Provincia"]

df_new_sel = df_new[id_cols_new + cols_100_new].copy()

df_new_long = df_new_sel.melt(
    id_vars=id_cols_new,
    value_vars=cols_100_new,
    var_name="periodo",
    value_name="cob_100mbps"
)
df_new_long["año"] = df_new_long["periodo"].str.extract(r"(\d{4})").astype(int)

df_new_agg = (df_new_long
    .groupby(["Comunidad Autónoma", "Provincia", "año"], as_index=False)["cob_100mbps"]
    .max()
)

print(f"✅ {f_new.name}")
print(f"   Años: {sorted(df_new_agg['año'].unique())}")
print(f"   Provincias: {df_new_agg['Provincia'].nunique()}")
print(df_new_agg.head())


✅ Cobertura_BA_España_2021-2024_MUN_PROV_CCAA_Nacional_datosgob.xlsx
   Años: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   Provincias: 52
  Comunidad Autónoma Provincia   año  cob_100mbps
0          ANDALUCÍA   Almería  2021     0.857542
1          ANDALUCÍA   Almería  2022     0.882811
2          ANDALUCÍA   Almería  2023     0.925456
3          ANDALUCÍA   Almería  2024     0.931974
4          ANDALUCÍA     Cádiz  2021     0.922155


In [3]:
# ── 3. CONCAT 2013-2024 ──────────────────────────────────────────────────────
df_concat = pd.concat([df_old_agg, df_new_agg], ignore_index=True)

# Eliminar duplicados si los hubiera (e.g., 2020 en ambos)
df_concat = df_concat.sort_values(["Provincia", "año"]).drop_duplicates(
    subset=["Provincia", "año"], keep="last"
)

# Normalizar nombre de provincia (strip + title)
df_concat["Provincia"] = df_concat["Provincia"].str.strip().str.title()

print(df_concat.shape)
print(df_concat["año"].unique())
print(df_concat.head(10))

df_concat.to_csv(OUT / "cobertura_provincia_año.csv", index=False)
print("✅ Guardado en data/conectividad/cobertura_provincia_año.csv")

(624, 4)
[2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
     Comunidad Autónoma Provincia   año  cob_100mbps
200  CASTILLA-LA MANCHA  Albacete  2013     0.525929
201  CASTILLA-LA MANCHA  Albacete  2014     0.529683
202  CASTILLA-LA MANCHA  Albacete  2015     0.537551
203  CASTILLA-LA MANCHA  Albacete  2016     0.536744
204  CASTILLA-LA MANCHA  Albacete  2017     0.621609
205  CASTILLA-LA MANCHA  Albacete  2018     0.746987
206  CASTILLA-LA MANCHA  Albacete  2019     0.885003
207  CASTILLA-LA MANCHA  Albacete  2020     0.920779
516  CASTILLA-LA MANCHA  Albacete  2021     0.875416
517  CASTILLA-LA MANCHA  Albacete  2022     0.900656
✅ Guardado en data/conectividad/cobertura_provincia_año.csv
